# Random Survival Forests (R)

Non-parametric ensemble survival model using `randomForestSRC::rfsrc()`.

| Python (scikit-survival) | R (randomForestSRC) |
|--------------------------|---------------------|
| `RandomSurvivalForest(n_estimators=1000)` | `rfsrc(ntree=1000)` |
| `min_samples_leaf=30` | `nodesize=30` |
| `max_features='log2'` | `mtry=floor(log2(p))` |
| `oob_score=True` | OOB error built-in |
| `permutation_importance` | `importance='permute'` |

**Libraries:** `randomForestSRC`, `survival`, `survminer`, `ggplot2`, `pheatmap`

In [ ]:
# install.packages(c('randomForestSRC','survival','survminer','ggplot2','dplyr','pheatmap','ggpubr'))

In [ ]:
suppressPackageStartupMessages({
  library(survival)
  library(randomForestSRC)
  library(survminer)
  library(ggplot2)
  library(dplyr)
  library(pheatmap)
  library(ggpubr)
})

DATA_DIR  <- '../../datasets/csv_files'
VIS_DIR   <- '../../visuals'
MODEL_DIR <- '../../models'
dir.create(VIS_DIR,   showWarnings=FALSE, recursive=TRUE)
dir.create(MODEL_DIR, showWarnings=FALSE, recursive=TRUE)
set.seed(42)
message('Packages loaded.')

## Preparing Data

In [ ]:
load_surv <- function(path) {
  df <- read.csv(path, stringsAsFactors=FALSE)
  df[complete.cases(df[c('relapse_free_time','relapse_free_event')]), ]
}

train_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/train_data.csv'))
test1_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_one.csv'))
test2_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_two.csv'))
test3_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_three.csv'))

surv_meta <- c('sample_name','relapse_free_time','relapse_free_event')
gene_cols <- setdiff(names(train_raw), surv_meta)
train_raw$relapse_free_event <- as.integer(train_raw$relapse_free_event)

cat(sprintf('Train: %d | Test1: %d | Test2: %d | Test3: %d | Genes: %d\n',
            nrow(train_raw), nrow(test1_raw), nrow(test2_raw),
            nrow(test3_raw), length(gene_cols)))
head(train_raw[, 1:6])

In [ ]:
prep_df <- function(df, genes) {
  avail <- intersect(genes, names(df))
  out   <- df[, c('relapse_free_time','relapse_free_event', avail)]
  out$relapse_free_event <- as.integer(out$relapse_free_event)
  out
}

train_df <- prep_df(train_raw, gene_cols)
test1_df <- prep_df(test1_raw, gene_cols)
test2_df <- prep_df(test2_raw, gene_cols)
test3_df <- prep_df(test3_raw, gene_cols)

## Fit Random Survival Forest

Parameters matched to Python notebook:
- `ntree=1000`, `nodesize=30` (min_samples_leaf), `nsplit=20` (min_samples_split proxy)
- `mtry=log2(p)`, `importance='permute'` (OOB permutation importance)

In [ ]:
cat('Fitting Random Survival Forest (ntree=1000)... this may take a minute\n')
rsf_model <- rfsrc(
  Surv(relapse_free_time, relapse_free_event) ~ .,
  data       = train_df,
  ntree      = 1000,
  nodesize   = 30,
  nsplit     = 20,
  mtry       = max(1L, floor(log2(length(gene_cols)))),
  importance = 'permute',
  seed       = -42
)
cat('Done.\n')
print(rsf_model)

## C-Index on All Datasets

In [ ]:
cindex_from_pred <- function(pred, time, event) {
  concordance(Surv(time, event) ~ pred$predicted)$concordance
}

train_ci <- 1 - rsf_model$err.rate[rsf_model$ntree]  # OOB C-index
pred_t1  <- predict(rsf_model, newdata=test1_df)
pred_t2  <- predict(rsf_model, newdata=test2_df)
pred_t3  <- predict(rsf_model, newdata=test3_df)

ci_t1 <- cindex_from_pred(pred_t1, test1_df$relapse_free_time, test1_df$relapse_free_event)
ci_t2 <- cindex_from_pred(pred_t2, test2_df$relapse_free_time, test2_df$relapse_free_event)
ci_t3 <- cindex_from_pred(pred_t3, test3_df$relapse_free_time, test3_df$relapse_free_event)

cat(sprintf('Train OOB C-index: %.4f\n', train_ci))
cat(sprintf('Test 1 C-index:    %.4f\n', ci_t1))
cat(sprintf('Test 2 C-index:    %.4f\n', ci_t2))
cat(sprintf('Test 3 C-index:    %.4f\n', ci_t3))

## Permutation Variable Importance

In [ ]:
vimp_df <- data.frame(
  gene       = names(rsf_model$importance),
  importance = as.vector(rsf_model$importance),
  stringsAsFactors = FALSE
)
vimp_df <- vimp_df[order(vimp_df$importance, decreasing=TRUE), ]

cat('Top 15 genes by permutation importance:\n')
print(head(vimp_df, 15))

top_n     <- 8
sig_genes <- head(vimp_df$gene, top_n)
cat(sprintf('\nRSF Gene Signature (%d genes): %s\n', top_n, paste(sig_genes, collapse=', ')))

PAPER_GENES <- c('TSLP','BIRC5','S100B','MDK','S100P','RARRES3','BLNK','ACO1')
overlap <- intersect(sig_genes, PAPER_GENES)
cat(sprintf('Paper overlap: %d/%d — %s\n', length(overlap),
            length(PAPER_GENES), paste(overlap, collapse=', ')))

In [ ]:
p_vimp <- ggplot(head(vimp_df, 20),
                 aes(x=importance, y=reorder(gene, importance))) +
  geom_col(fill='#4DBBD5', alpha=0.8) +
  geom_vline(xintercept=0, colour='grey40') +
  labs(title='RSF — Permutation Variable Importance (Top 20)',
       x='Mean decrease in C-index', y=NULL) +
  theme_bw(base_size=10) +
  theme(plot.title=element_text(face='bold'))

p_vimp
ggsave(file.path(VIS_DIR, 'rsf_permutation_importance.png'), p_vimp,
       dpi=150, width=8, height=6)
cat('Saved: rsf_permutation_importance.png\n')

## Patient Stratification & Kaplan–Meier Curves

In [ ]:
train_risk      <- rsf_model$predicted.oob
train_med_risk  <- median(train_risk)
risk_label      <- function(risk, cutoff) ifelse(risk >= cutoff, 'High Risk', 'Low Risk')

make_km_df <- function(risk, df) {
  data.frame(
    time       = df$relapse_free_time,
    event      = as.integer(df$relapse_free_event),
    risk_group = factor(risk_label(risk, train_med_risk),
                        levels=c('Low Risk','High Risk'))
  )
}

km_plot <- function(df_km, title) {
  fit <- survfit(Surv(time, event) ~ risk_group, data=df_km)
  ggsurvplot(fit, data=df_km, pval=TRUE, conf.int=TRUE,
             palette=c('Low Risk'='#4DBBD5','High Risk'='#E64B35'),
             title=title, xlab='Time (days)', ylab='Relapse-Free Survival',
             legend.labs=c('Low Risk','High Risk'),
             ggtheme=theme_bw(base_size=9))$plot +
    theme(plot.title=element_text(face='bold', size=10))
}

km_plots <- list(
  km_plot(make_km_df(train_risk,       train_df), 'Train (OOB)'),
  km_plot(make_km_df(pred_t1$predicted, test1_df), 'Test 1'),
  km_plot(make_km_df(pred_t2$predicted, test2_df), 'Test 2'),
  km_plot(make_km_df(pred_t3$predicted, test3_df), 'Test 3')
)
ggarrange(plotlist=km_plots, ncol=4, common.legend=TRUE, legend='bottom')

## Gene Signature Heatmap

In [ ]:
sort_idx      <- order(train_risk)
sorted_groups <- risk_label(train_risk[sort_idx], train_med_risk)
expr_mat      <- as.matrix(train_df[sort_idx, sig_genes, drop=FALSE])
expr_z        <- t(scale(t(expr_mat)))
rownames(expr_z) <- sig_genes
n_pts <- nrow(expr_mat)
colnames(expr_z) <- paste0('P', seq_len(n_pts))

ann_col    <- data.frame(Risk_Group=sorted_groups, row.names=colnames(expr_z))
ann_colors <- list(Risk_Group=c('High Risk'='#E64B35','Low Risk'='#4DBBD5'))

pheatmap(expr_z, cluster_cols=FALSE, cluster_rows=FALSE,
         annotation_col=ann_col, annotation_colors=ann_colors,
         color=colorRampPalette(c('#4DBBD5','white','#E64B35'))(100),
         breaks=seq(-2,2,length.out=101),
         show_colnames=FALSE, fontsize_row=10,
         main='RSF Gene Signature — Expression Heatmap (sorted by risk score)')

In [ ]:
long_df <- do.call(rbind, lapply(sig_genes, function(g)
  data.frame(gene=g, expression=train_df[[g]],
             risk_group=risk_label(train_risk, train_med_risk),
             stringsAsFactors=FALSE)))

ggplot(long_df, aes(x=risk_group, y=expression, fill=risk_group)) +
  geom_violin(alpha=0.65, trim=FALSE) +
  geom_jitter(width=0.08, alpha=0.4, size=1.0) +
  stat_summary(fun=median, geom='crossbar', width=0.45, colour='black', fatten=1.5) +
  scale_fill_manual(values=c('High Risk'='#E64B35','Low Risk'='#4DBBD5'), guide='none') +
  facet_wrap(~gene, scales='free_y', nrow=2) +
  labs(title='RSF Signature Gene Expression: High vs Low Risk',
       x=NULL, y='Expression (log2)') +
  theme_bw(base_size=9) +
  theme(plot.title=element_text(face='bold'), strip.text=element_text(face='bold'))

## Save Model & Results

In [ ]:
saveRDS(rsf_model, file.path(MODEL_DIR, 'rsf_model.rds'))
write.csv(vimp_df, file.path(DATA_DIR,  'rsf_importance.csv'), row.names=FALSE)
write.csv(
  data.frame(sample_name=train_raw$sample_name,
             risk_score =train_risk,
             risk_group =risk_label(train_risk, train_med_risk)),
  file.path(DATA_DIR, 'rsf_risk_scores_train.csv'), row.names=FALSE)

cat(sprintf('\nC-index summary:\n  Train OOB %.4f | Test1 %.4f | Test2 %.4f | Test3 %.4f\n',
            train_ci, ci_t1, ci_t2, ci_t3))
cat('Saved: rsf_model.rds, rsf_importance.csv, rsf_risk_scores_train.csv\n')